# Hybrid RAG with ChromaDB + Neo4j

> **Teaching goal:** this notebook is designed as a step-by-step, instructor-friendly Colab lab. Every important cell explains what it does, why it matters, and how it fits into RAG / GraphRAG.

## Corpus used in this lab
We use public research papers as a realistic mini-corpus:

1. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al.
2. **From Local to Global: A Graph RAG Approach to Query-Focused Summarization** — Edge et al.

These papers are useful because they naturally contain entities, methods, datasets, claims, and relationships. That makes them good for comparing:

- **Traditional RAG:** retrieve semantically similar chunks.
- **GraphRAG:** retrieve facts and relationships from a knowledge graph.
- **Hybrid RAG:** combine vector retrieval and graph traversal.

## What this notebook demonstrates

Hybrid RAG combines two retrieval styles:

| Retrieval style | Best for |
|---|---|
| Vector retrieval | semantic passage lookup |
| Graph retrieval | entity relationships and multi-hop reasoning |
| Hybrid retrieval | questions needing both text evidence and graph structure |

This notebook uses:

- **ChromaDB** for file-based vector search.
- **Neo4j** for graph facts.
- An LLM router to decide whether a query needs vector, graph, or both.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# In Colab, run this cell first. It installs:
# - openai: LLM + embeddings
# - pymupdf: PDF text extraction
# - pandas/numpy/tqdm: data handling
# - tiktoken: optional token-aware splitting

!pip -q install openai pymupdf pandas numpy tqdm tiktoken

!pip -q install chromadb neo4j

In [ ]:
# ============================================================
# 2. Configure API keys safely
# ============================================================
# Recommended in Colab:
#   Left sidebar -> Secrets -> add OPENAI_API_KEY
# Then this cell will read it without exposing the key in the notebook.

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    # Not running in Colab. You can set your key in your environment instead.
    pass

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in Colab Secrets or as an environment variable.")

from openai import OpenAI
client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("OpenAI client configured. Models:", CHAT_MODEL, EMBED_MODEL)

In [ ]:
# ============================================================
# 3. Common helper functions: LLM + embeddings
# ============================================================
# llm() is used for answer generation, extraction, routing, and evaluation.
# embed_texts() converts a list of strings into embedding vectors.

import numpy as np

def llm(prompt, system="You are a precise, helpful assistant.", temperature=0.0):
    """Single-turn LLM call. Returns plain text."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content.strip()


def embed_texts(texts, batch_size=64):
    """Embed a string or list of strings using OpenAI embeddings."""
    if isinstance(texts, str):
        texts = [texts]
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in response.data])
    return np.array(vectors, dtype="float32")

print(llm("Reply with exactly: ready"))
print("Embedding shape for one text:", embed_texts("hello").shape)

In [ ]:
# ============================================================
# 4. Download public research papers and extract text
# ============================================================
# We download PDFs from public URLs and extract text with PyMuPDF.
# The resulting `documents` list will be used by RAG, GraphRAG, and Hybrid RAG.

import requests, fitz, re
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path("/content/rag_graphrag_corpus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = [
    {
        "doc_id": "rag_lewis_2020",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url": "https://arxiv.org/pdf/2005.11401",
    },
    {
        "doc_id": "graphrag_edge_2024",
        "title": "From Local to Global: A Graph RAG Approach to Query-Focused Summarization",
        "url": "https://arxiv.org/pdf/2404.16130",
    },
]


def download_pdf(url, path):
    if path.exists() and path.stat().st_size > 10_000:
        return path
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def extract_pdf_text(path):
    doc = fitz.open(path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"page": page_num, "text": text})
    return pages

# Build a page-level document list.
documents = []
for paper in tqdm(PAPERS):
    pdf_path = DATA_DIR / f"{paper['doc_id']}.pdf"
    download_pdf(paper["url"], pdf_path)
    for page in extract_pdf_text(pdf_path):
        documents.append({
            "doc_id": paper["doc_id"],
            "title": paper["title"],
            "page": page["page"],
            "text": page["text"],
        })

print(f"Loaded {len(documents)} page-documents")
print(documents[0].keys())
print(documents[0]["title"], "page", documents[0]["page"])
print(documents[0]["text"][:500])

In [ ]:
# ============================================================
# 5. Chunk the corpus
# ============================================================
# Why chunk?
# LLMs and vector databases work better with small passages than entire PDFs.
# Each chunk keeps metadata so we can cite the source paper and page.

import textwrap


def chunk_text(text, chunk_size=900, overlap=150):
    """Simple character-based chunking with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = []
for d in documents:
    for j, chunk in enumerate(chunk_text(d["text"])):
        chunks.append({
            "id": f"{d['doc_id']}_p{d['page']}_c{j}",
            "text": chunk,
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "chunk_id": j,
        })

print(f"Created {len(chunks)} chunks")
print(chunks[0]["id"])
print(textwrap.shorten(chunks[0]["text"], width=600))

## Build the Chroma vector database

This is the same traditional RAG foundation: chunks are embedded and stored in a persistent Chroma collection.

In [ ]:
# ============================================================
# 6. Build or load Chroma vector DB
# ============================================================

import chromadb
from tqdm.auto import tqdm

CHROMA_DIR = "/content/chroma_hybrid_rag"
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_or_create_collection("hybrid_research_papers")

if collection.count() == 0:
    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [{"doc_id": c["doc_id"], "title": c["title"], "page": c["page"], "chunk_id": c["chunk_id"]} for c in chunks]
    embeddings = embed_texts(texts).tolist()
    for i in tqdm(range(0, len(chunks), 100)):
        collection.add(
            ids=ids[i:i+100],
            documents=texts[i:i+100],
            metadatas=metadatas[i:i+100],
            embeddings=embeddings[i:i+100],
        )
print("Chroma rows:", collection.count())

## Connect to Neo4j and create/load graph facts

For a classroom demo, this notebook can create graph facts from the corpus just like the GraphRAG notebook. In production, you would usually build this graph offline and update it as new documents arrive.

In [ ]:
# ============================================================
# 7. Connect to Neo4j
# ============================================================

from neo4j import GraphDatabase
import os, json, re

try:
    from google.colab import userdata
    NEO4J_URI = userdata.get("NEO4J_URI") or os.environ.get("NEO4J_URI", "")
    NEO4J_USERNAME = userdata.get("NEO4J_USERNAME") or os.environ.get("NEO4J_USERNAME", "neo4j")
    NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD") or os.environ.get("NEO4J_PASSWORD", "")
except Exception:
    NEO4J_URI = os.environ.get("NEO4J_URI", "")
    NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
    NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("Neo4j not configured. Graph retrieval cells will be skipped until credentials are set.")
    driver = None
else:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    print("Neo4j connected")

In [ ]:
# ============================================================
# 8. Extract and load graph triples for hybrid retrieval
# ============================================================

EXTRACT_SYS = "You extract knowledge graph triples. Return ONLY valid JSON."
EXTRACT_PROMPT = """
Extract relationship triples from the text.
Return JSON: {"triples": [{"source":"...", "relation":"...", "target":"...", "evidence":"..."}]}
Use canonical entity names. Relations should be short snake_case.

Text:
{text}
"""


def extract_triples(chunk):
    raw = llm(EXTRACT_PROMPT.format(text=chunk["text"]), system=EXTRACT_SYS)
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        triples = json.loads(raw).get("triples", [])
    except Exception:
        triples = []
    out = []
    for tr in triples:
        s = str(tr.get("source", "")).strip()
        r = str(tr.get("relation", "")).strip().lower().replace(" ", "_")
        t = str(tr.get("target", "")).strip()
        if s and r and t and s.lower() != t.lower():
            out.append({
                "source": s, "relation": r, "target": t,
                "evidence": str(tr.get("evidence", ""))[:500],
                "doc_id": chunk["doc_id"], "title": chunk["title"], "page": chunk["page"],
                "chunk_id": chunk["chunk_id"], "chunk_text": chunk["text"][:1500],
            })
    return out

MAX_EXTRACTION_CHUNKS = 20
all_triples = []
for ch in tqdm(chunks[:MAX_EXTRACTION_CHUNKS]):
    all_triples.extend(extract_triples(ch))

print("Triples extracted:", len(all_triples))

if driver is not None:
    with driver.session() as session:
        session.run("CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")
        session.run("MATCH (n:Entity) DETACH DELETE n")
        session.run("""
        UNWIND $rows AS row
        MERGE (s:Entity {name: row.source})
        MERGE (t:Entity {name: row.target})
        MERGE (s)-[r:RELATES_TO {relation: row.relation, doc_id: row.doc_id, page: row.page, chunk_id: row.chunk_id}]->(t)
        SET r.evidence = row.evidence, r.title = row.title, r.chunk_text = row.chunk_text
        """, rows=all_triples)
    print("Loaded graph into Neo4j")
else:
    print("Neo4j not configured, but triples are available in all_triples.")

## Build vector, graph, and hybrid retrievers

The hybrid system has three components:

1. `vector_retrieve()` gets semantically relevant chunks from Chroma.
2. `graph_retrieve()` gets relationship facts from Neo4j.
3. `route_query()` decides whether to use vector, graph, or both.

In [ ]:
# ============================================================
# 9. Retrieval functions
# ============================================================

import re

def vector_retrieve(query, k=5):
    qv = embed_texts(query)[0].tolist()
    res = collection.query(
        query_embeddings=[qv],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc, meta, distance in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "metadata": meta, "distance": distance})
    return hits


def graph_entity_candidates(question, max_results=10):
    if driver is None:
        return []
    tokens = [t for t in re.findall(r"[A-Za-z][A-Za-z0-9_-]+", question) if len(t) > 2]
    if not tokens:
        return []
    with driver.session() as session:
        rows = session.run("""
        MATCH (e:Entity)
        WHERE any(tok IN $tokens WHERE toLower(e.name) CONTAINS toLower(tok))
        RETURN e.name AS name
        LIMIT $limit
        """, tokens=tokens, limit=max_results).data()
    return [r["name"] for r in rows]


def graph_retrieve(question, hops=2, limit=60):
    if driver is None:
        return []
    entities = graph_entity_candidates(question)
    if not entities:
        return []
    with driver.session() as session:
        rows = session.run("""
        MATCH (seed:Entity)
        WHERE seed.name IN $entities
        MATCH path=(seed)-[*1..2]-(n)
        WITH relationships(path) AS rels
        UNWIND rels AS r
        WITH DISTINCT startNode(r).name AS source,
             r.relation AS relation,
             endNode(r).name AS target,
             r.doc_id AS doc_id,
             r.page AS page,
             r.evidence AS evidence
        RETURN source, relation, target, doc_id, page, evidence
        LIMIT $limit
        """, entities=entities, limit=limit).data()
    return rows

In [ ]:
# ============================================================
# 10. Query router
# ============================================================
# The router chooses the retrieval mode.
# For a production system, you can replace this with a classifier, rules, or evaluation-based routing.

ROUTER_SYS = "You are a query router for a RAG system. Return only one label: vector, graph, or hybrid."
ROUTER_PROMPT = """
Choose the best retrieval strategy for this question.

Use vector when the question asks for definitions, explanations, summaries, or local text evidence.
Use graph when the question asks about relationships, paths, connections, dependencies, ownership, authorship, or multi-hop links.
Use hybrid when the question needs both textual evidence and relationship structure.

Question: {query}
Label:
"""


def route_query(query):
    label = llm(ROUTER_PROMPT.format(query=query), system=ROUTER_SYS).strip().lower()
    if "hybrid" in label:
        return "hybrid"
    if "graph" in label:
        return "graph"
    return "vector"

for q in [
    "What is RAG?",
    "How is GraphRAG related to knowledge graphs?",
    "Explain GraphRAG and cite both text evidence and relationship facts.",
]:
    print(q, "->", route_query(q))

## Hybrid answer function

The final answer prompt can include:

- Vector context: raw passages from papers.
- Graph context: structured relationships from Neo4j.

This gives the LLM both semantic evidence and relationship evidence.

In [ ]:
# ============================================================
# 11. Unified answer function
# ============================================================

def format_vector_context(hits):
    lines = []
    for i, h in enumerate(hits, start=1):
        m = h["metadata"]
        lines.append(f"[V{i}: {m['doc_id']} p.{m['page']}] {h['text']}")
    return "\n\n".join(lines)


def format_graph_context(facts):
    return "\n".join(
        f"[G{i}] {f['source']} -[{f['relation']}]-> {f['target']} (source: {f['doc_id']} p.{f['page']}; evidence: {f.get('evidence','')})"
        for i, f in enumerate(facts, start=1)
    )


def hybrid_answer(query, mode="auto", k=5):
    if mode == "auto":
        mode = route_query(query)

    vector_hits = vector_retrieve(query, k=k) if mode in ["vector", "hybrid"] else []
    graph_facts = graph_retrieve(query) if mode in ["graph", "hybrid"] else []

    vector_context = format_vector_context(vector_hits) if vector_hits else "No vector context used."
    graph_context = format_graph_context(graph_facts) if graph_facts else "No graph context used."

    prompt = f"""
Answer the question using the supplied retrieval context.
Use vector context for textual evidence and graph context for relationships.
If evidence is incomplete, say what is missing.
Cite vector sources as [V1], [V2] and graph facts as [G1], [G2].

Retrieval mode: {mode}

Vector context:
{vector_context}

Graph context:
{graph_context}

Question: {query}
Answer:
"""
    answer = llm(prompt)

    print("Mode:", mode)
    print("Vector hits:", len(vector_hits))
    print("Graph facts:", len(graph_facts))
    print("\nAnswer:\n", answer)
    return answer, {"mode": mode, "vector_hits": vector_hits, "graph_facts": graph_facts}

_ = hybrid_answer("Why does GraphRAG help with questions that naive vector RAG can struggle with?", mode="hybrid")

## Compare modes on the same questions

This section is useful for teaching. It shows that no single retrieval method is always best.

In [ ]:
# ============================================================
# 12. Side-by-side comparison
# ============================================================

TEST_QUESTIONS = [
    "What is Retrieval-Augmented Generation?",
    "How is GraphRAG related to knowledge graphs and community summaries?",
    "Why can naive vector RAG struggle with global questions over a corpus?",
]

for q in TEST_QUESTIONS:
    print("\n" + "="*110)
    print("QUESTION:", q)
    print("\n--- VECTOR ---")
    hybrid_answer(q, mode="vector", k=4)
    print("\n--- GRAPH ---")
    hybrid_answer(q, mode="graph", k=4)
    print("\n--- HYBRID ---")
    hybrid_answer(q, mode="hybrid", k=4)

## Teaching recap

Hybrid RAG workflow:

```text
Question
  ├─ router says vector → Chroma chunks → LLM
  ├─ router says graph  → Neo4j facts → LLM
  └─ router says hybrid → Chroma chunks + Neo4j facts → LLM
```

Main advantage:

- More flexible than either vector-only or graph-only retrieval.

Main limitation:

- More moving parts: two indexes, extraction quality, routing, and answer synthesis.